# λ_d Phase 2 — Colab
**Runtime → Change runtime → T4 GPU**

Run cell below. Interrupt (Runtime → Interrupt) to save checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
CKPT_DIR = '/content/drive/MyDrive/lambda_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints → {CKPT_DIR}')

# Download code (no git auth needed)
REPO = '/content/EVA-Ai'
if not os.path.exists(REPO):
    !wget -q -O /content/repo.zip https://github.com/BlackCatSpb/EVA-Ai/archive/main.zip
    !unzip -q -o /content/repo.zip -d /content/
    !mv /content/EVA-Ai-main {REPO}
    !rm -f /content/repo.zip
os.chdir(REPO)
sys.path.insert(0, '.')
print(f'Code: {REPO}')

# Install + data
!pip install -q datasets tokenizers
CACHE = 'wikitext_chunks.npy'
if not os.path.exists(CACHE):
    import numpy as np
    from datasets import load_dataset
    from transformers import AutoTokenizer
    ds = load_dataset('Salesforce/wikitext','wikitext-103-raw-v1',split='train',streaming=True)
    tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
    all_tok = []
    for i, ex in enumerate(ds):
        all_tok.extend(tok(ex['text'])['input_ids'])
        if len(all_tok) >= 80000*128: break
        if i%10000==0: print(f'  {i} ex, {len(all_tok)} tok')
    n = len(all_tok)//128
    np.array(all_tok[:n*128],dtype=np.int32).reshape(n,128).dump(CACHE)
    print(f'Data: {n} chunks')
else:
    import numpy as np
    print(f'Data cached: {np.load(CACHE).shape[0]} chunks')

# Train
print()
!python colab_train.py --ckpt_dir "{CKPT_DIR}" --batch_size 8 --n_train 50000